# **IA/KNN**

## **Treinamento suculenta**

### **Gerando dados para suculenta**

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
def gerar_dados_suculenta(quantidade_amostras=2000):
    np.random.seed(42) # mantém os dados iguais toda vez que rodar, significa que os dados são aleatórios mas não tanto

    # gerando as 4 variáveis dos sensores (dados brutos) baseado na modelagem de dados
    umidade_solo_atual = np.random.uniform(0, 1023, quantidade_amostras) # gera dados da umidade do solo, 0=molhado e 1023=seco
    umidade_solo_ha_7_dias = np.random.uniform(0, 1023, quantidade_amostras) # gera dados da umidade do solo de 7 dias atrás

    # gerando a variável de contexto (já convertida em números)
    # estacao_ano: 1 = Verão, 2 = Primavera, 3 = Inverno, 4 = Outono
    estacao_ano = np.random.choice([1, 2, 3, 4], size=quantidade_amostras)

    niveis_saude = []

    # 3. aplicando a lógica de combinação para definir a saúde (o que a IA vai aprender)
    for i in range(quantidade_amostras):
        # cálculo da variação da umidade para análise preditiva
        variacao_umidade = umidade_solo_atual[i] - umidade_solo_ha_7_dias[i]

        # CRÍTICO: solo encharcado hoje (<300) e já estava encharcado há 7 dias (variação menor que 100) #no inverno e outono (3 e 4)
        if umidade_solo_atual[i] < 300 and abs(variacao_umidade) < 100: #and (estacao_ano[i] in [3, 4]):
            niveis_saude.append("Crítico")

        # RUIM: solo secou rápido demais (variação > 500)
        elif variacao_umidade > 500:
            niveis_saude.append("Ruim")

        # EXCELENTE: solo na faixa seca ideal para suculenta (650 a 900) e estável/secando no verão/primavera com boa luz
        elif (650 <= umidade_solo_atual[i] <= 900) and variacao_umidade >= 0:
            niveis_saude.append("Excelente")

        # BOM: solo em faixas aceitáveis sem extremos
        elif (550 <= umidade_solo_atual[i] <= 850):
            niveis_saude.append("Bom")

        # RAZOÁVEL: se não quebrou nenhuma regra e não entra em nenhuma condição, fica no meio termo
        else:
            niveis_saude.append("Razoável")

    # Criando o DataFrame do Pandas
    df = pd.DataFrame({
        'umidade_solo_atual': umidade_solo_atual,
        'umidade_solo_ha_7_dias': umidade_solo_ha_7_dias,
        'estacao_ano': estacao_ano,
        'saude_planta': niveis_saude
    })

    return df

#### **Executando a função e criando um DataFrame**

In [ ]:
# executa a função para criar o dataframe com todos os dados
dados_suculenta = gerar_dados_suculenta(2000)

# mostra a distribuição das notas de saúde para ver se ficou equilibrado
print("Distribuição das classes de saúde:")
print(dados_suculenta['saude_planta'].value_counts())

# exibe as 5 primeiras linhas da tabela criada
# 'head()' é pessoalmente um comando que eu gosto muito!!
dados_suculenta.head()

Distribuição das classes de saúde:
saude_planta
Razoável     1082
Bom           305
Ruim          269
Excelente     230
Crítico       114
Name: count, dtype: int64


,umidade_solo_atual,umidade_solo_ha_7_dias,estacao_ano,saude_planta
0,383.154542,267.724914,2,Razoável
1,972.580735,252.659311,1,Ruim
2,748.829802,927.098436,4,Bom
3,612.427629,255.285762,1,Bom
4,159.607069,278.204570,3,Razoável


### **Separando os dados treinamento/teste para suculenta**

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
# 1. separando as variáveis preditoras (X) da variável alvo (y)
x_suculenta = dados_suculenta.drop(columns=['saude_planta']) # tudo menos a resposta
y_suculenta = dados_suculenta['saude_planta'] # apenas a resposta

# 2. dividindo em treino (80%) e teste (20%)
x_treino_suculenta, x_teste_suculenta, y_treino_suculenta, y_teste_suculenta = train_test_split(
    x_suculenta, y_suculenta,
    test_size=0.20, # separa 20% para teste
    random_state=42, # garante que a divisão seja igual toda vez que rodar
    stratify=y_suculenta # mantém a mesma proporção de notas de saúde no treino e no teste (crítico, bom, etc)
)

print(f"Quantidade de dados para o Lee estudar (Treino): {len(x_treino_suculenta)}")
print(f"Quantidade de dados para a prova do Lee (Teste): {len(x_teste_suculenta)}")

Quantidade de dados para o Lee estudar (Treino): 1600
Quantidade de dados para a prova do Lee (Teste): 400


### **Padronização dos dados**

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
# cria o objeto que fará a padronização
scaler_suculenta = StandardScaler()

# o 'fit_transform' faz duas coisas:
## calcula a média/desvio usando o x_treino
## aplica a transformação matemática nele
x_treino_suculenta_normalizado = scaler_suculenta.fit_transform(x_treino_suculenta)

# no x_teste, usamos APENAS o 'transform': ele usa a mesma média que aprendeu com o treino, mantendo a prova totalmente inédita para o Lee
x_teste_suculenta_normalizado = scaler_suculenta.transform(x_teste_suculenta)

print("Exemplo da primeira linha do treino original:\n", x_treino_suculenta.iloc[0].values)
print("\nExemplo da primeira linha do treino normalizado:\n", x_treino_suculenta_normalizado[0])

Exemplo da primeira linha do treino original:
 [225.84731388 120.25498005   3.        ]

Exemplo da primeira linha do treino normalizado:
 [-0.94624175 -1.33035366  0.45247548]


### **Treinamento com base na suculenta**

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [ ]:
# 1. criamos o Lee definindo o número de vizinhos (K=5 é um bom padrão)
# abaixo de 3 é muito baixo e influenciável (overfitting)
# acima de 10 pode perder detalhes (underfitting)
lee_suculenta = KNeighborsClassifier(n_neighbors=5)

# 2. o Lee estuda os dados de treino normalizados
lee_suculenta.fit(x_treino_suculenta_normalizado, y_treino_suculenta)

# 3. o Lee faz a prova com os dados de teste que ele nunca viu
previsoes_suculenta = lee_suculenta.predict(x_teste_suculenta_normalizado)

# 4. calculamos quanto de acerto o Lee teve no teste (Acurácia)
precisao_suculenta = accuracy_score(y_teste_suculenta, previsoes_suculenta)
print(f"Precisão do Lee para a Suculenta: {precisao_suculenta * 100:.2f}%")

Precisão do Lee para a Suculenta: 92.50%


In [ ]:
teste_suculenta = np.array([
    [30.0,   40.0,   1],
    [90.0,   150.0,  2],
    [180.0,  220.0,  3],
    [320.0,  280.0,  4],
    [450.0,  500.0,  1],
    [620.0,  400.0,  2],
    [700.0,  800.0,  3],
    [850.0,  650.0,  4],
    [920.0,  950.0,  1],
    [990.0,  980.0,  2],
    [0.0,    0.0,    1],
    [1000.0, 0.0,    2],
    [0.0,    1000.0, 3],
    [1000.0, 1000.0, 4],
    [50.0,   950.0,  1],
    [950.0,  50.0,   2],
    [150.0,  850.0,  3],
    [850.0,  150.0,  4],
    [500.0,  500.0,  1],
    [999.0,  1.0,    2]
])

teste_suculenta_norm = scaler_suculenta.transform(teste_suculenta)

previsao_suculenta = lee_suculenta.predict(teste_suculenta_norm)

display(teste_suculenta)
display(teste_suculenta_norm)

for resultado in previsao_suculenta:
    print(f"Estado Classificado: {resultado}")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


array([[  30.,   40.,    1.],
       [  90.,  150.,    2.],
       [ 180.,  220.,    3.],
       [ 320.,  280.,    4.],
       [ 450.,  500.,    1.],
       [ 620.,  400.,    2.],
       [ 700.,  800.,    3.],
       [ 850.,  650.,    4.],
       [ 920.,  950.,    1.],
       [ 990.,  980.,    2.],
       [   0.,    0.,    1.],
       [1000.,    0.,    2.],
       [   0., 1000.,    3.],
       [1000., 1000.,    4.],
       [  50.,  950.,    1.],
       [ 950.,   50.,    2.],
       [ 150.,  850.,    3.],
       [ 850.,  150.,    4.],
       [ 500.,  500.,    1.],
       [ 999.,    1.,    2.]])

array([[-1.60176236, -1.60479958, -1.36196253],
       [-1.40093634, -1.22863537, -0.45474353],
       [-1.09969731, -0.98925815,  0.45247548],
       [-0.63110326, -0.78407767,  1.35969449],
       [-0.19598021, -0.03174925, -1.36196253],
       [ 0.37302685, -0.37371671, -0.45474353],
       [ 0.64079488,  0.99415314,  0.45247548],
       [ 1.14285993,  0.48120194,  1.35969449],
       [ 1.37715695,  1.50710433, -1.36196253],
       [ 1.61145398,  1.60969457, -0.45474353],
       [-1.70217537, -1.74158657, -1.36196253],
       [ 1.64492498, -1.74158657, -0.45474353],
       [-1.70217537,  1.67808807,  0.45247548],
       [ 1.64492498,  1.67808807,  1.35969449],
       [-1.53482035,  1.50710433, -1.36196253],
       [ 1.47756997, -1.57060284, -0.45474353],
       [-1.20011032,  1.16513687,  0.45247548],
       [ 1.14285993, -1.22863537,  1.35969449],
       [-0.02862519, -0.03174925, -1.36196253],
       [ 1.64157788, -1.73816689, -0.45474353]])

Estado Classificado: Crítico
Estado Classificado: Crítico
Estado Classificado: Crítico
Estado Classificado: Crítico
Estado Classificado: Razoável
Estado Classificado: Bom
Estado Classificado: Bom
Estado Classificado: Excelente
Estado Classificado: Razoável
Estado Classificado: Razoável
Estado Classificado: Crítico
Estado Classificado: Ruim
Estado Classificado: Razoável
Estado Classificado: Razoável
Estado Classificado: Razoável
Estado Classificado: Ruim
Estado Classificado: Razoável
Estado Classificado: Ruim
Estado Classificado: Razoável
Estado Classificado: Ruim


## **Treinamento salsinha**

### **Gerando dados**

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
def gerar_dados_salsinha(quantidade_amostras=2000):
    np.random.seed(42)

    umidade_solo_atual = np.random.uniform(0, 1023, quantidade_amostras)
    umidade_solo_ha_7_dias = np.random.uniform(0, 1023, quantidade_amostras)
    estacao_ano = np.random.choice([1, 2, 3, 4], size=quantidade_amostras)

    niveis_saude = []

    for i in range(quantidade_amostras):
        variacao_umidade = umidade_solo_atual[i] - umidade_solo_ha_7_dias[i]

        # CRÍTICO: solo muito seco (acima de 600) e secou muito nos últimos 7 dias (variação > 200)
        if umidade_solo_atual[i] > 600 and variacao_umidade > 200:
            niveis_saude.append("Crítico")

        # RUIM: solo começou a secar (perto de 500)
        elif 400 < umidade_solo_atual[i] <= 600:
            niveis_saude.append("Ruim")

        # EXCELENTE: solo bem úmido (100 a 400) e variação controlada (abaixo de 50, sem secar bruscamente)
        elif (100 <= umidade_solo_atual[i] <= 400) and variacao_umidade < 50:
            niveis_saude.append("Excelente")

        # BOM: solo aceitável
        elif (400 < umidade_solo_atual[i] <= 500):
            niveis_saude.append("Bom")
        else:
            niveis_saude.append("Razoável")

    return pd.DataFrame({
        'umidade_solo_atual': umidade_solo_atual,
        'umidade_solo_ha_7_dias': umidade_solo_ha_7_dias,
        'estacao_ano': estacao_ano,
        'saude_planta': niveis_saude
    })

In [ ]:
dados_salsinha = gerar_dados_salsinha(2000)
display(dados_salsinha.head())

,umidade_solo_atual,umidade_solo_ha_7_dias,estacao_ano,saude_planta
0,383.154542,267.724914,2,Razoável
1,972.580735,252.659311,1,Crítico
2,748.829802,927.098436,4,Razoável
3,612.427629,255.285762,1,Crítico
4,159.607069,278.204570,3,Excelente


### **Separando dados treinamento/teste**

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
# 1. separando as variáveis preditoras (x) da variável alvo (y)
x_salsinha = dados_salsinha.drop(columns=['saude_planta'])
y_salsinha = dados_salsinha['saude_planta']

# 2. Divisão clássica 80/20
x_treino_salsinha, x_teste_salsinha, y_treino_salsinha, y_teste_salsinha = train_test_split(
    x_salsinha, y_salsinha,
    test_size=0.20,
    random_state=42,
    stratify=y_salsinha
)

print(f"Quantidade de dados para o Lee estudar (Treino): {len(x_treino_salsinha)}")
print(f"Quantidade de dados para a prova do Lee (Teste): {len(x_teste_salsinha)}")

Quantidade de dados para o Lee estudar (Treino): 1600
Quantidade de dados para a prova do Lee (Teste): 400


### **Padronização dos dados**

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [ ]:
from sklearn.preprocessing import StandardScaler

# 1. padronizador exclusivo para a Salsinha
scaler_salsinha = StandardScaler()

# 2. 'fit_transform' calcula os limites do treino e já aplica a escala nas 1600 amostras
x_treino_salsinha_normalizado = scaler_salsinha.fit_transform(x_treino_salsinha)

# 3. 'transform' aplica a MESMA régua do treino nas 400 amostras da prova (teste)
x_teste_salsinha_normalizado = scaler_salsinha.transform(x_teste_salsinha)

print("Exemplo da primeira linha do treino original:\n", x_treino_salsinha.iloc[0].values)
print("\nExemplo da primeira linha do treino normalizado:\n", x_treino_salsinha_normalizado[0])

Exemplo da primeira linha do treino original:
 [542.49908231  86.63617374   1.        ]

Exemplo da primeira linha do treino normalizado:
 [ 0.10659685 -1.4419703  -1.32264169]


### **Treinamento e testes**

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [ ]:
# 1. "cérebro" do Lee para a salsinha (K=5)
lee_salsinha = KNeighborsClassifier(n_neighbors=5)

# 2. o Lee estuda os dados de treino normalizados da salsinha
lee_salsinha.fit(x_treino_salsinha_normalizado, y_treino_salsinha)

# 3. o Lee faz a prova com os dados de teste que ele nunca viu antes
previsoes_salsinha = lee_salsinha.predict(x_teste_salsinha_normalizado)

# 4. calculamos quanto de acerto o Lee teve no teste (Acurácia)
precisao_salsinha = accuracy_score(y_teste_salsinha, previsoes_salsinha)
print(f"Precisão do Lee para a Salsinha: {precisao_suculenta * 100:.2f}%")

Precisão do Lee para a Salsinha: 92.50%


In [ ]:
teste_salsinha = np.array([
    [100.0,  120.0,  1],
    [220.0,  260.0,  2],
    [340.0,  300.0,  3],
    [450.0,  500.0,  4],
    [580.0,  620.0,  1],
    [650.0,  700.0,  2],
    [750.0,  680.0,  3],
    [850.0,  900.0,  4],
    [930.0,  950.0,  1],
    [1000.0, 980.0,  2],
    [5.0,    5.0,    1],
    [1000.0, 10.0,   2],
    [10.0,   1000.0, 3],
    [1000.0, 1000.0, 4],
    [100.0,  900.0,  1],
    [900.0,  100.0,  2],
    [250.0,  800.0,  3],
    [800.0,  250.0,  4],
    [500.0,  500.0,  1],
    [750.0,  50.0,   2]
])

teste_salsinha_norm = scaler_salsinha.transform(teste_salsinha)

previsao_salsinha = lee_salsinha.predict(teste_salsinha_norm)

display(teste_salsinha)
display(teste_salsinha_norm)

for resultado in previsao_salsinha:
    print(f"Estado Classificado: {resultado}")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


array([[ 100.,  120.,    1.],
       [ 220.,  260.,    2.],
       [ 340.,  300.,    3.],
       [ 450.,  500.,    4.],
       [ 580.,  620.,    1.],
       [ 650.,  700.,    2.],
       [ 750.,  680.,    3.],
       [ 850.,  900.,    4.],
       [ 930.,  950.,    1.],
       [1000.,  980.,    2.],
       [   5.,    5.,    1.],
       [1000.,   10.,    2.],
       [  10., 1000.,    3.],
       [1000., 1000.,    4.],
       [ 100.,  900.,    1.],
       [ 900.,  100.,    2.],
       [ 250.,  800.,    3.],
       [ 800.,  250.,    4.],
       [ 500.,  500.,    1.],
       [ 750.,   50.,    2.]])

array([[-1.36572812, -1.32863818, -1.32264169],
       [-0.96645272, -0.85307831, -0.42896487],
       [-0.56717733, -0.71720407,  0.46471195],
       [-0.20117488, -0.03783282,  1.35838876],
       [ 0.23137347,  0.36978992, -1.32264169],
       [ 0.46428412,  0.64153842, -0.42896487],
       [ 0.79701362,  0.57360129,  0.46471195],
       [ 1.12974311,  1.32090966,  1.35838876],
       [ 1.39592671,  1.49075247, -1.32264169],
       [ 1.62883736,  1.59265816, -0.42896487],
       [-1.68182114, -1.71927665, -1.32264169],
       [ 1.62883736, -1.70229237, -0.42896487],
       [-1.66518467,  1.66059528,  0.46471195],
       [ 1.62883736,  1.66059528,  1.35838876],
       [-1.36572812,  1.32090966, -1.32264169],
       [ 1.29610786, -1.39657531, -0.42896487],
       [-0.86663387,  0.98122404,  0.46471195],
       [ 0.96337836, -0.88704688,  1.35838876],
       [-0.03481013, -0.03783282, -1.32264169],
       [ 0.79701362, -1.56641812, -0.42896487]])

Estado Classificado: Razoável
Estado Classificado: Excelente
Estado Classificado: Excelente
Estado Classificado: Ruim
Estado Classificado: Ruim
Estado Classificado: Razoável
Estado Classificado: Razoável
Estado Classificado: Razoável
Estado Classificado: Razoável
Estado Classificado: Razoável
Estado Classificado: Razoável
Estado Classificado: Crítico
Estado Classificado: Razoável
Estado Classificado: Razoável
Estado Classificado: Excelente
Estado Classificado: Crítico
Estado Classificado: Excelente
Estado Classificado: Crítico
Estado Classificado: Ruim
Estado Classificado: Crítico


In [ ]:
print(y_salsinha.value_counts())

saude_planta
Razoável     673
Crítico      502
Excelente    454
Ruim         371
Name: count, dtype: int64


## **Treinamento cebolinha**

### **Gerando dados**

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
def gerar_dados_cebolinha(quantidade_amostras=2000):
    np.random.seed(42)

    umidade_solo_atual = np.random.uniform(0, 1023, quantidade_amostras)
    umidade_solo_ha_7_dias = np.random.uniform(0, 1023, quantidade_amostras)
    estacao_ano = np.random.choice([1, 2, 3, 4], size=quantidade_amostras)

    niveis_saude = []

    for i in range(quantidade_amostras):
        variacao_umidade = umidade_solo_atual[i] - umidade_solo_ha_7_dias[i]

        # CRÍTICO: solo extremamente seco (acima de 700) e subindo rápido (variação > 250)
        # significa que ela está torrando no seco total
        if umidade_solo_atual[i] > 700 and variacao_umidade > 250:
            niveis_saude.append("Crítico")

        # RUIM: solo bem seco (entre 500 e 700)
        elif 500 < umidade_solo_atual[i] <= 700:
            niveis_saude.append("Ruim")

        # EXCELENTE: solo úmido e aceitável (150 a 500) e variação estável (abaixo de 80)
        elif (150 <= umidade_solo_atual[i] <= 500) and variacao_umidade < 80:
            niveis_saude.append("Excelente")

        # BOM: solo ligeiramente mais úmido ou seco, mas contornável
        elif (100 <= umidade_solo_atual[i] < 150) or (500 < umidade_solo_atual[i] <= 550):
            niveis_saude.append("Bom")

        # RAZOÁVEL: meio termo
        else:
            niveis_saude.append("Razoável")

    return pd.DataFrame({
        'umidade_solo_atual': umidade_solo_atual,
        'umidade_solo_ha_7_dias': umidade_solo_ha_7_dias,
        'estacao_ano': estacao_ano,
        'saude_planta': niveis_saude
    })

In [ ]:
dados_cebolinha = gerar_dados_cebolinha(2000)

display(dados_cebolinha.head())

,umidade_solo_atual,umidade_solo_ha_7_dias,estacao_ano,saude_planta
0,383.154542,267.724914,2,Razoável
1,972.580735,252.659311,1,Crítico
2,748.829802,927.098436,4,Razoável
3,612.427629,255.285762,1,Ruim
4,159.607069,278.204570,3,Excelente


### **Separando dados treinamento/teste**

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
# 1. separando as variáveis preditoras (X) da variável alvo (y)
x_cebolinha = dados_cebolinha.drop(columns=['saude_planta']) # tudo menos a resposta
y_cebolinha = dados_cebolinha['saude_planta'] # apenas a resposta

# 2. dividindo em treino (80%) e teste (20%)
x_treino_cebolinha, x_teste_cebolinha, y_treino_cebolinha, y_teste_cebolinha = train_test_split(
    x_cebolinha, y_cebolinha,
    test_size=0.20, # separa 20% para teste
    random_state=42, # garante que a divisão seja igual toda vez que rodar
    stratify=y_cebolinha # mantém a mesma proporção de notas de saúde no treino e no teste (crítico, bom, etc)
)

print(f"Quantidade de dados para o Lee estudar (Treino): {len(x_treino_cebolinha)}")
print(f"Quantidade de dados para a prova do Lee (Teste): {len(x_teste_cebolinha)}")

Quantidade de dados para o Lee estudar (Treino): 1600
Quantidade de dados para a prova do Lee (Teste): 400


### **Padronização dos dados**

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
# cria o objeto que fará a padronização
scaler_cebolinha = StandardScaler()

# 'fit_transform' calcula os limites do treino e já aplica a escala nas 1600 amostras
x_treino_cebolinha_normalizado = scaler_cebolinha.fit_transform(x_treino_cebolinha)

# 'transform' aplica a MESMA régua do treino nas 400 amostras da prova (teste)
x_teste_cebolinha_normalizado = scaler_cebolinha.transform(x_teste_cebolinha)

print("Exemplo da primeira linha do treino original:\n", x_treino_cebolinha.iloc[0].values)
print("\nExemplo da primeira linha do treino normalizado:\n", x_treino_cebolinha_normalizado[0])

Exemplo da primeira linha do treino original:
 [406.11209197 629.27711077   1.        ]

Exemplo da primeira linha do treino normalizado:
 [-0.34689316  0.42091075 -1.34838846]


### **Treinamento e testes**

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [ ]:
# 1. criamos o Lee definindo o número de vizinhos (K=5 é um bom padrão)
lee_cebolinha = KNeighborsClassifier(n_neighbors=5)

# 2. o Lee estuda os dados de treino normalizados
lee_cebolinha.fit(x_treino_cebolinha_normalizado, y_treino_cebolinha)

# 3. o Lee faz a prova com os dados de teste que ele nunca viu
previsoes_cebolinha = lee_cebolinha.predict(x_teste_cebolinha_normalizado)

# 4. calculamos quanto de acerto o Lee teve no teste (Acurácia)
precisao_cebolinha = accuracy_score(y_teste_cebolinha, previsoes_cebolinha)
print(f"Precisão do Lee para a Cebolinha: {precisao_cebolinha * 100:.2f}%")

Precisão do Lee para a Cebolinha: 92.25%


In [ ]:
teste_2_cebolinha = np.array([
    [80.0,   100.0,  1],
    [180.0,  250.0,  2],
    [320.0,  280.0,  3],
    [450.0,  500.0,  4],
    [550.0,  620.0,  1],
    [680.0,  700.0,  2],
    [750.0,  820.0,  3],
    [850.0,  900.0,  4],
    [930.0,  960.0,  1],
    [995.0,  990.0,  2],
    [0.0,    0.0,    1],
    [1000.0, 0.0,    2],
    [0.0,    1000.0, 3],
    [1000.0, 1000.0, 4],
    [25.0,   975.0,  1],
    [975.0,  25.0,   2],
    [150.0,  850.0,  3],
    [850.0,  150.0,  4],
    [500.0,  500.0,  1],
    [1.0,    999.0,  2]
])

teste_2_cebolinha_norm = scaler_cebolinha.transform(teste_2_cebolinha)

display(teste_2_cebolinha)
display(teste_2_cebolinha_norm)

previsao_teste_2_cebolinha = lee_cebolinha.predict(teste_2_cebolinha_norm)

for resultado in previsao_teste_2_cebolinha:
    print(f"Estado Classificado: {resultado}")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


array([[  80.,  100.,    1.],
       [ 180.,  250.,    2.],
       [ 320.,  280.,    3.],
       [ 450.,  500.,    4.],
       [ 550.,  620.,    1.],
       [ 680.,  700.,    2.],
       [ 750.,  820.,    3.],
       [ 850.,  900.,    4.],
       [ 930.,  960.,    1.],
       [ 995.,  990.,    2.],
       [   0.,    0.,    1.],
       [1000.,    0.,    2.],
       [   0., 1000.,    3.],
       [1000., 1000.,    4.],
       [  25.,  975.,    1.],
       [ 975.,   25.,    2.],
       [ 150.,  850.,    3.],
       [ 850.,  150.,    4.],
       [ 500.,  500.,    1.],
       [   1.,  999.,    2.]])

array([[-1.43339173, -1.36991784, -1.34838846],
       [-1.10022452, -0.86238732, -0.44532377],
       [-0.63379042, -0.76088121,  0.45774091],
       [-0.20067304, -0.01650311,  1.3608056 ],
       [ 0.13249417,  0.3895213 , -1.34838846],
       [ 0.56561154,  0.66020425, -0.44532377],
       [ 0.79882859,  1.06622866,  0.45774091],
       [ 1.13199581,  1.33691161,  1.3608056 ],
       [ 1.39852958,  1.53992381, -1.34838846],
       [ 1.61508826,  1.64142992, -0.44532377],
       [-1.6999255 , -1.70827152, -1.34838846],
       [ 1.63174662, -1.70827152, -0.44532377],
       [-1.6999255 ,  1.67526529,  0.45774091],
       [ 1.63174662,  1.67526529,  1.3608056 ],
       [-1.6166337 ,  1.59067687, -1.34838846],
       [ 1.54845482, -1.6236831 , -0.44532377],
       [-1.20017468,  1.16773477,  0.45774091],
       [ 1.13199581, -1.200741  ,  1.3608056 ],
       [-0.03408944, -0.01650311, -1.34838846],
       [-1.69659383,  1.67188175, -0.44532377]])

Estado Classificado: Bom
Estado Classificado: Excelente
Estado Classificado: Excelente
Estado Classificado: Excelente
Estado Classificado: Ruim
Estado Classificado: Ruim
Estado Classificado: Razoável
Estado Classificado: Razoável
Estado Classificado: Razoável
Estado Classificado: Razoável
Estado Classificado: Razoável
Estado Classificado: Crítico
Estado Classificado: Razoável
Estado Classificado: Razoável
Estado Classificado: Razoável
Estado Classificado: Crítico
Estado Classificado: Bom
Estado Classificado: Crítico
Estado Classificado: Ruim
Estado Classificado: Razoável


## **Treinamento lírio-da-paz**

### **Gerando dados**

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
def gerar_dados_lirio_paz(quantidade_amostras=2000):
    np.random.seed(42) # mantém os dados iguais toda vez que rodar, significa que os dados são aleatórios mas não tanto

    # gerando as 4 variáveis dos sensores (dados brutos) baseado na modelagem de dados
    umidade_solo_atual = np.random.uniform(0, 1023, quantidade_amostras) # gera dados da umidade do solo, 0=molhado e 1023=seco
    umidade_solo_ha_7_dias = np.random.uniform(0, 1023, quantidade_amostras) # gera dados da umidade do solo de 7 dias atrás

    # gerando a variável de contexto (já convertida em números)
    # estacao_ano: 1 = Verão, 2 = Primavera, 3 = Inverno, 4 = Outono
    estacao_ano = np.random.choice([1, 2, 3, 4], size=quantidade_amostras)

    niveis_saude = []
    for i in range(quantidade_amostras):
        variacao_umidade = umidade_solo_atual[i] - umidade_solo_ha_7_dias[i]

        # CRÍTICO: planta murcha por sede (solo passou de 550)
        if umidade_solo_atual[i] > 750:
            niveis_saude.append("Crítico")

        # RUIM: excesso de água/falta de drenagem (solo < 150 e quase sem variação positiva, ou seja, continua encharcado)
        elif umidade_solo_atual[i] < 150 and variacao_umidade < 50:
            niveis_saude.append("Ruim")

        # EXCELENTE: o equilíbrio perfeito (solo úmido entre 150 e 400, bem drenado)
        elif (200 <= umidade_solo_atual[i] <= 450) and (abs(variacao_umidade) < 250):
            niveis_saude.append("Excelente")

        # BOM: faixa aceitável de transição (400 a 550) antes de murchar
        elif (450 < umidade_solo_atual[i] <= 650):
            niveis_saude.append("Bom")

        # RAZOÁVEL: meio termo
        else:
            niveis_saude.append("Razoável")

    # criando o DataFrame do Pandas
    df = pd.DataFrame({
        'umidade_solo_atual': umidade_solo_atual,
        'umidade_solo_ha_7_dias': umidade_solo_ha_7_dias,
        'estacao_ano': estacao_ano,
        'saude_planta': niveis_saude
    })

    return df

In [ ]:
# executa a função para criar o dataframe com todos os dados
dados_lirio_paz = gerar_dados_lirio_paz(2000)

# mostra a distribuição das notas de saúde para ver se ficou equilibrado
print("Distribuição das classes de saúde:")
print(dados_lirio_paz['saude_planta'].value_counts())

# exibe as 5 primeiras linhas da tabela criada
dados_lirio_paz.head()

Distribuição das classes de saúde:
saude_planta
Razoável     535
Crítico      530
Bom          390
Ruim         305
Excelente    240
Name: count, dtype: int64


,umidade_solo_atual,umidade_solo_ha_7_dias,estacao_ano,saude_planta
0,383.154542,267.724914,2,Excelente
1,972.580735,252.659311,1,Crítico
2,748.829802,927.098436,4,Razoável
3,612.427629,255.285762,1,Bom
4,159.607069,278.204570,3,Razoável


### **Separando dados treinamento/teste**

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
# 1. separando as variáveis preditoras (X) da variável alvo (y)
x_lirio_paz = dados_lirio_paz.drop(columns=['saude_planta']) # tudo menos a resposta
y_lirio_paz = dados_lirio_paz['saude_planta'] # apenas a resposta

# 2. dividindo em treino (80%) e teste (20%)
x_treino_lirio_paz, x_teste_lirio_paz, y_treino_lirio_paz, y_teste_lirio_paz = train_test_split(
    x_lirio_paz, y_lirio_paz,
    test_size=0.20, # separa 20% para teste
    random_state=42, # garante que a divisão seja igual toda vez que rodar
    stratify=y_lirio_paz # mantém a mesma proporção de notas de saúde no treino e no teste (crítico, bom, etc)
)

print(f"Quantidade de dados para o Lee estudar (Treino): {len(x_treino_lirio_paz)}")
print(f"Quantidade de dados para a prova do Lee (Teste): {len(x_teste_lirio_paz)}")

Quantidade de dados para o Lee estudar (Treino): 1600
Quantidade de dados para a prova do Lee (Teste): 400


### **Padronização dos dados**

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
# cria o objeto que fará a padronização
scaler_lirio_paz = StandardScaler()

# 'fit_transform' calcula os limites do treino e já aplica a escala nas 1600 amostras
x_treino_lirio_paz_normalizado = scaler_lirio_paz.fit_transform(x_treino_lirio_paz)

# 'transform' aplica a MESMA régua do treino nas 400 amostras da prova (teste)
x_teste_lirio_paz_normalizado = scaler_lirio_paz.transform(x_teste_lirio_paz)

print("Exemplo da primeira linha do treino original:\n", x_treino_lirio_paz.iloc[0].values)
print("\nExemplo da primeira linha do treino normalizado:\n", x_treino_lirio_paz_normalizado[0])

Exemplo da primeira linha do treino original:
 [287.39600341  53.15157884   2.        ]

Exemplo da primeira linha do treino normalizado:
 [-0.75025978 -1.53917646 -0.46250973]


### **Treinamento e testes**

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [ ]:
# 1. criamos o Lee definindo o número de vizinhos (K=5 é um bom padrão)
lee_lirio_paz = KNeighborsClassifier(n_neighbors=5)

# 2. o Lee estuda os dados de treino normalizados
lee_lirio_paz.fit(x_treino_lirio_paz_normalizado, y_treino_lirio_paz)

# 3. o Lee faz a prova com os dados de teste que ele nunca viu
previsoes_lirio_paz = lee_lirio_paz.predict(x_teste_lirio_paz_normalizado)

# 4. calculamos quanto de acerto o Lee teve no teste (Acurácia)
precisao_lirio_paz = accuracy_score(y_teste_lirio_paz, previsoes_lirio_paz)
print(f"Precisão do Lee para Lírio da Paz: {precisao_lirio_paz * 100:.2f}%")

Precisão do Lee para Lírio da Paz: 91.75%


In [ ]:
teste_lirio_paz = np.array([
    [15.0,   20.0,   1],  # extremamente seco
    [75.0,   120.0,  3],  # seco
    [220.0,  180.0,  2],  # umidade baixa
    [450.0,  470.0,  4],  # intermediário
    [600.0,  580.0,  1],  # intermediário alto
    [350.0,  900.0,  2],  # secou recentemente
    [850.0,  300.0,  3],  # recebeu muita água recentemente
    [920.0,  950.0,  4],  # extremamente úmido
    [1000.0, 1000.0, 1],  # máximo possível
    [50.0,   980.0,  2],  # caso extremo e contraditório
    [0.0,    0.0,    1],  # começa com valores muito absurdos
    [1000.0, 0.0,    2],
    [0.0,    1000.0, 3],
    [1000.0, 1000.0, 4],
    [50.0,   950.0,  1],
    [950.0,  50.0,   2],
    [200.0,  800.0,  3],
    [800.0,  200.0,  4],
    [500.0,  500.0,  1],
    [750.0,  750.0,  2]
])

teste_lirio_paz_norm = scaler_lirio_paz.transform(teste_lirio_paz)

previsao_lirio_paz = lee_lirio_paz.predict(teste_lirio_paz_norm)

display(teste_lirio_paz)
display(teste_lirio_paz_norm)

for resultado in previsao_lirio_paz:
    print(f"Estado Classificado: {resultado}")

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


array([[  15.,   20.,    1.],
       [  75.,  120.,    3.],
       [ 220.,  180.,    2.],
       [ 450.,  470.,    4.],
       [ 600.,  580.,    1.],
       [ 350.,  900.,    2.],
       [ 850.,  300.,    3.],
       [ 920.,  950.,    4.],
       [1000., 1000.,    1.],
       [  50.,  980.,    2.],
       [   0.,    0.,    1.],
       [1000.,    0.,    2.],
       [   0., 1000.,    3.],
       [1000., 1000.,    4.],
       [  50.,  950.,    1.],
       [ 950.,   50.,    2.],
       [ 200.,  800.,    3.],
       [ 800.,  200.,    4.],
       [ 500.,  500.,    1.],
       [ 750.,  750.,    2.]])

array([[-1.66272122, -1.65140577, -1.36606964],
       [-1.46173558, -1.31287193,  0.44105018],
       [-0.97602027, -1.10975163, -0.46250973],
       [-0.20557529, -0.12800352,  1.34461009],
       [ 0.29688882,  0.2443837 , -1.36606964],
       [-0.54055137,  1.32769197, -0.46250973],
       [ 1.13432901, -0.70351103,  0.44105018],
       [ 1.36881226,  1.49695889,  1.34461009],
       [ 1.63679312,  1.6662258 , -1.36606964],
       [-1.5454796 ,  1.59851904, -0.46250973],
       [-1.71296763, -1.71911253, -1.36606964],
       [ 1.63679312, -1.71911253, -0.46250973],
       [-1.71296763,  1.6662258 ,  0.44105018],
       [ 1.63679312,  1.6662258 ,  1.34461009],
       [-1.5454796 ,  1.49695889, -1.36606964],
       [ 1.46930508, -1.54984562, -0.46250973],
       [-1.04301548,  0.98915814,  0.44105018],
       [ 0.96684097, -1.04204487,  1.34461009],
       [-0.03808726, -0.02644337, -1.36606964],
       [ 0.79935293,  0.81989122, -0.46250973]])

Estado Classificado: Ruim
Estado Classificado: Ruim
Estado Classificado: Excelente
Estado Classificado: Bom
Estado Classificado: Bom
Estado Classificado: Razoável
Estado Classificado: Crítico
Estado Classificado: Crítico
Estado Classificado: Crítico
Estado Classificado: Ruim
Estado Classificado: Ruim
Estado Classificado: Crítico
Estado Classificado: Ruim
Estado Classificado: Crítico
Estado Classificado: Ruim
Estado Classificado: Crítico
Estado Classificado: Razoável
Estado Classificado: Crítico
Estado Classificado: Bom
Estado Classificado: Crítico


## **Dados gerais para treino**

In [ ]:
import numpy as np

In [ ]:
teste_geral = np.array([

    # ===== MUITO SECOS =====
    [900, 850, 1],
    [950, 900, 2],
    [1000, 950, 3],
    [850, 800, 4],
    [920, 870, 1],
    [980, 920, 2],
    [870, 820, 3],
    [940, 890, 4],
    [1000, 1000, 1],
    [800, 780, 2],
    [750, 700, 3],
    [820, 760, 4],
    [970, 940, 1],
    [890, 850, 2],
    [930, 910, 3],

    # ===== MUITO ÚMIDOS =====
    [20, 30, 1],
    [50, 40, 2],
    [80, 70, 3],
    [10, 20, 4],
    [100, 90, 1],
    [40, 50, 2],
    [120, 110, 3],
    [60, 80, 4],
    [0, 0, 1],
    [150, 120, 2],
    [90, 100, 3],
    [30, 20, 4],
    [70, 60, 1],
    [110, 130, 2],
    [140, 100, 3],

    # ===== INTERMEDIÁRIOS =====
    [250, 240, 1],
    [300, 280, 2],
    [350, 340, 3],
    [400, 390, 4],
    [450, 430, 1],
    [500, 480, 2],
    [550, 520, 3],
    [600, 580, 4],
    [320, 300, 1],
    [380, 350, 2],
    [420, 400, 3],
    [470, 450, 4],
    [520, 500, 1],
    [580, 550, 2],
    [650, 620, 3],

    # ===== CONTRADITÓRIOS =====
    [50, 950, 1],
    [950, 50, 2],
    [100, 900, 3],
    [900, 100, 4],
    [200, 800, 1],
    [800, 200, 2],
    [300, 700, 3],
    [700, 300, 4],
    [400, 950, 1],
    [950, 400, 2],
    [150, 850, 3],
    [850, 150, 4],
    [250, 750, 1],
    [750, 250, 2],
    [500, 1000, 3]

])

In [ ]:
# normalização dos dados
teste_geral_suculenta_norm = scaler_suculenta.transform(teste_geral)
teste_geral_salsinha_norm = scaler_salsinha.transform(teste_geral)
teste_geral_cebolinha_norm = scaler_cebolinha.transform(teste_geral)
teste_geral_lirio_paz_norm = scaler_lirio_paz.transform(teste_geral)

# resultados dos modelos
previsao_geral_suculenta = lee_lirio_paz.predict(teste_geral_suculenta_norm)
previsao_geral_salsinha = lee_salsinha.predict(teste_geral_salsinha_norm)
previsao_geral_cebolinha = lee_cebolinha.predict(teste_geral_cebolinha_norm)
previsao_geral_lirio_paz = lee_lirio_paz.predict(teste_geral_lirio_paz_norm)

# saídas
print("Array de dados para teste:")
display(teste_geral)

## teste SUCULENTA
print("\nTeste suculenta:")
#display(teste_geral_suculenta_norm)

for resultado in previsao_geral_suculenta:
    print(f"Estado Classificado: {resultado}")

## teste SALSINHA
print("\nTeste salsinha:")
#display(teste_geral_salsinha_norm)

for resultado in previsao_geral_salsinha:
    print(f"Estado Classificado: {resultado}")

## teste CEBOLINHA
print("\nTeste cebolinha:")
#display(teste_geral_cebolinha_norm)

for resultado in previsao_geral_cebolinha:
    print(f"Estado Classificado: {resultado}")

## teste LÍRIO-DA-PAZ
print("\nTeste lírio-da-paz:")
#display(teste_geral_lirio_paz_norm)

for resultado in previsao_geral_lirio_paz:
    print(f"Estado Classificado: {resultado}")

Array de dados para teste:


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


array([[ 900,  850,    1],
       [ 950,  900,    2],
       [1000,  950,    3],
       [ 850,  800,    4],
       [ 920,  870,    1],
       [ 980,  920,    2],
       [ 870,  820,    3],
       [ 940,  890,    4],
       [1000, 1000,    1],
       [ 800,  780,    2],
       [ 750,  700,    3],
       [ 820,  760,    4],
       [ 970,  940,    1],
       [ 890,  850,    2],
       [ 930,  910,    3],
       [  20,   30,    1],
       [  50,   40,    2],
       [  80,   70,    3],
       [  10,   20,    4],
       [ 100,   90,    1],
       [  40,   50,    2],
       [ 120,  110,    3],
       [  60,   80,    4],
       [   0,    0,    1],
       [ 150,  120,    2],
       [  90,  100,    3],
       [  30,   20,    4],
       [  70,   60,    1],
       [ 110,  130,    2],
       [ 140,  100,    3],
       [ 250,  240,    1],
       [ 300,  280,    2],
       [ 350,  340,    3],
       [ 400,  390,    4],
       [ 450,  430,    1],
       [ 500,  480,    2],
       [ 550,  520,    3],
 


Teste suculenta:
Estado Classificado: Crítico
Estado Classificado: Crítico
Estado Classificado: Crítico
Estado Classificado: Crítico
Estado Classificado: Crítico
Estado Classificado: Crítico
Estado Classificado: Crítico
Estado Classificado: Crítico
Estado Classificado: Crítico
Estado Classificado: Crítico
Estado Classificado: Crítico
Estado Classificado: Crítico
Estado Classificado: Crítico
Estado Classificado: Crítico
Estado Classificado: Crítico
Estado Classificado: Ruim
Estado Classificado: Ruim
Estado Classificado: Ruim
Estado Classificado: Ruim
Estado Classificado: Ruim
Estado Classificado: Ruim
Estado Classificado: Ruim
Estado Classificado: Ruim
Estado Classificado: Ruim
Estado Classificado: Razoável
Estado Classificado: Ruim
Estado Classificado: Ruim
Estado Classificado: Ruim
Estado Classificado: Ruim
Estado Classificado: Ruim
Estado Classificado: Excelente
Estado Classificado: Excelente
Estado Classificado: Excelente
Estado Classificado: Excelente
Estado Classificado: Bom
Esta

# **Anotação final geral**

Foram desenvolvidos quatro modelos de classificação utilizando o algoritmo K-Nearest Neighbors (KNN), cada um especializado em uma espécie vegetal: cebolinha, salsinha, suculenta e lírio-da-paz. Os modelos alcançaram índices de precisão iguais e superiores a 90% e foram submetidos a testes de estresse compostos por sessenta cenários distintos, incluindo condições extremas de seca, excesso de umidade, situações intermediárias e combinações contraditórias entre umidade atual e histórica. Os resultados mostraram que os modelos foram capazes de identificar padrões específicos de cada planta, produzindo classificações coerentes e diferenciadas mesmo quando expostos às mesmas entradas. Destacou-se o modelo da cebolinha pela melhor distribuição das classes e capacidade de adaptação, enquanto a suculenta apresentou o comportamento mais consistente. O modelo do lírio-da-paz exigiu ajustes nas regras de geração dos dados para reduzir o desbalanceamento das classes, resultando em respostas mais realistas. De forma geral, os experimentos indicam que a solução desenvolvida apresenta confiabilidade e potencial para aplicação em sistemas IoT de monitoramento inteligente de plantas.

# **Exportando arquivo `.pkl`**

In [1]:
import pickle

In [ ]:
# salvar o modelo treinado dentro do arquivo
with open('lee_suculenta.pkl', 'wb') as arquivo:
    pickle.dump(lee_suculenta, arquivo) # o primeiro parâmetro é o nome da variável em que salvamos o modelo KNN

print("Modelo salvo com sucesso como 'lee_suculenta.pkl'!")

In [ ]:
# salvar o modelo treinado dentro do arquivo
with open('lee_salsinha.pkl', 'wb') as arquivo:
    pickle.dump(lee_salsinha, arquivo) #

print("Modelo salvo com sucesso como 'lee_salsinha.pkl'!")

In [ ]:
# salvar o modelo treinado dentro do arquivo
with open('lee_cebolinha.pkl', 'wb') as arquivo:
    pickle.dump(lee_cebolinha, arquivo)

print("Modelo salvo com sucesso como 'lee_cebolinha.pkl'!")

In [ ]:
# salvar o modelo treinado dentro do arquivo
with open('lee_lirio_paz.pkl', 'wb') as arquivo:
    pickle.dump(lee_lirio_paz, arquivo)

print("Modelo salvo com sucesso como 'lee_lirio_paz.pkl'!")

In [ ]:
from google.colab import files

# download do gcolab pro desktop
files.download('lee_suculenta.pkl')
files.download('lee_salsinha.pkl')
files.download('lee_cebolinha.pkl')
files.download('lee_lirio_paz.pkl')